In [1]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv
import graphviz
import json
from graphviz import Digraph

In [2]:
load_dotenv()

True

In [3]:
llm = ChatOpenAI(model="gpt-3.5-turbo-0125")

In [4]:
text = """
In 2017, a young engineer named Arjun Mehta started working at Tesla as a battery research intern. During his internship, he contributed to the development of 4680 battery cells, which later improved the Model Y’s range by 15%.
In 2019, after graduating from IIT Bombay, Arjun joined Tesla full-time as a Battery Systems Engineer. He collaborated closely with Dr. Lisa Wong, a materials scientist, to optimize the anode chemistry using silicon nanowires.
Their research paper titled “Silicon Nanowire Anodes for High-Energy Batteries” was published in Nature Energy in 2020, gaining widespread recognition.
In 2021, Arjun moved to SpaceX to work on Starship’s thermal protection systems, applying his materials expertise.
By 2023, he founded his own startup NanoVolt Energy, aiming to commercialize solid-state lithium batteries for electric aviation.
Today, NanoVolt has raised $50 million in Series A funding led by Sequoia Capital and is working with companies like Joby Aviation to integrate these batteries into their eVTOL aircraft.
"""

In [12]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["text"],
    template="""
You are an advanced Knowledge Graph Extraction AI.

Your task is to read the given text and extract all entities and their relationships in the form of structured JSON triples. Each triple should have:

- subject: The main entity or actor.
- relation: The relationship or action connecting subject and object.
- object: The target entity or item acted upon.

### Instructions:
- Capture people, organizations, dates, events, technologies, products, etc.
- Keep relation names concise, using verbs like "worked at", "founded", "published in", "collaborated with", "developed", "improved", "graduated from", "raised", "partnered with".
- If an entity is associated with a date or year, include it as a separate triple with relation “date” or integrate it into the event triple if contextually meaningful.

### Output formatting rules:
- Return ONLY a valid JSON array of triples.
- Do NOT include markdown formatting (no ```json or ```).
- Do NOT include any explanation text, comments, or additional messages.
- Ensure the JSON is parsable directly without further cleaning.

### Text to process:

{text}
"""
)

final_prompt = prompt_template.format(text=text)


In [13]:
response = llm.invoke(final_prompt)
print(response)

content='[\n    {"subject": "Arjun Mehta", "relation": "worked at", "object": "Tesla", "date": "2017"},\n    {"subject": "Arjun Mehta", "relation": "contribute to", "object": "development of 4680 battery cells"},\n    {"subject": "4680 battery cells", "relation": "improved", "object": "Model Y’s range"},\n    {"subject": "Arjun Mehta", "relation": "graduated from", "object": "IIT Bombay", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "joined", "object": "Tesla full-time", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "collaborated with", "object": "Dr. Lisa Wong"},\n    {"subject": "Dr. Lisa Wong", "relation": "worked at", "object": "Tesla"},\n    {"subject": "Arjun Mehta", "relation": "optimized", "object": "anode chemistry using silicon nanowires"},\n    {"subject": "research paper", "relation": "titled", "object": "Silicon Nanowire Anodes for High-Energy Batteries"},\n    {"subject": "research paper", "relation": "published in", "object": "Nature Energy

In [14]:
type(response.content)

str

In [15]:
response.content

'[\n    {"subject": "Arjun Mehta", "relation": "worked at", "object": "Tesla", "date": "2017"},\n    {"subject": "Arjun Mehta", "relation": "contribute to", "object": "development of 4680 battery cells"},\n    {"subject": "4680 battery cells", "relation": "improved", "object": "Model Y’s range"},\n    {"subject": "Arjun Mehta", "relation": "graduated from", "object": "IIT Bombay", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "joined", "object": "Tesla full-time", "date": "2019"},\n    {"subject": "Arjun Mehta", "relation": "collaborated with", "object": "Dr. Lisa Wong"},\n    {"subject": "Dr. Lisa Wong", "relation": "worked at", "object": "Tesla"},\n    {"subject": "Arjun Mehta", "relation": "optimized", "object": "anode chemistry using silicon nanowires"},\n    {"subject": "research paper", "relation": "titled", "object": "Silicon Nanowire Anodes for High-Energy Batteries"},\n    {"subject": "research paper", "relation": "published in", "object": "Nature Energy", "date

In [16]:
json.loads(response.content)

[{'subject': 'Arjun Mehta',
  'relation': 'worked at',
  'object': 'Tesla',
  'date': '2017'},
 {'subject': 'Arjun Mehta',
  'relation': 'contribute to',
  'object': 'development of 4680 battery cells'},
 {'subject': '4680 battery cells',
  'relation': 'improved',
  'object': 'Model Y’s range'},
 {'subject': 'Arjun Mehta',
  'relation': 'graduated from',
  'object': 'IIT Bombay',
  'date': '2019'},
 {'subject': 'Arjun Mehta',
  'relation': 'joined',
  'object': 'Tesla full-time',
  'date': '2019'},
 {'subject': 'Arjun Mehta',
  'relation': 'collaborated with',
  'object': 'Dr. Lisa Wong'},
 {'subject': 'Dr. Lisa Wong', 'relation': 'worked at', 'object': 'Tesla'},
 {'subject': 'Arjun Mehta',
  'relation': 'optimized',
  'object': 'anode chemistry using silicon nanowires'},
 {'subject': 'research paper',
  'relation': 'titled',
  'object': 'Silicon Nanowire Anodes for High-Energy Batteries'},
 {'subject': 'research paper',
  'relation': 'published in',
  'object': 'Nature Energy',
  'dat

In [17]:
kg_str = response.content

# Parse string to JSON
kg = json.loads(kg_str)
# Initialize Graphviz directed graph
dot = Digraph(comment='Knowledge Graph', format='png')
dot.attr('node', shape='ellipse')

# Add edges from triples
for triple in kg:
    subj = triple['subject']
    obj = triple['object']
    rel = triple['relation']
    
    label = rel
    # If 'date' field exists, append to relation label
    if 'date' in triple:
        label += f" ({triple['date']})"
    
    dot.edge(subj, obj, label=label)

# Render and open the graph
dot.render('knowledge_graph', view=True)


'knowledge_graph.png'